# 1. PROJECT CONTEXT

## Research Title
**"Analisis Performa Algoritma Machine Learning untuk Klasifikasi Jenis dan Tingkat Keparahan Cyberbullying pada Teks Bahasa Indonesia Menggunakan TF-IDF"**
*(Performance Analysis of Machine Learning Algorithms for Cyberbullying Type and Severity Classification in Indonesian Text Using TF-IDF)*

## Role of Hyperparameter Tuning
This is the ninth stage of the pipeline. While Baseline Models (Stage 08) give us a starting point, Hyperparameter Tuning aims to find the optimal configuration for each algorithm to maximize its predictive performance on the unseen text data.

## Zero Data Leakage Principle
To prevent optimistic bias (data leakage), the tuning process strictly utilizes **only the training dataset**. We will use *Cross-Validation* (K-Fold) to simulate training and validation loops *internally* within the training data. The actual Test and Validation sets will remain completely isolated.


# 2. HYPERPARAMETER TUNING OBJECTIVES

Our goals for this stage are to:
- Improve the configuration of Logistic Regression, Linear SVC, and XGBoost models.
- Identify optimal hyperparameter combinations systematically using Grid Search and Randomized Search.
- Use `f1_macro` as our primary scoring metric to handle class imbalance fairly.
- Save the most optimized (tuned) versions of the models for the final evaluation stage.


In [1]:
# 3. IMPORT LIBRARIES

import pandas as pd
import numpy as np
import pathlib
import time
import json
import joblib
from scipy import sparse
import warnings

# Sklearn Tuning Modules
from sklearn.model_selection import StratifiedKFold, GridSearchCV, RandomizedSearchCV

# Sklearn Algorithms (to define base estimators if not loading from disk)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
import xgboost as xgb

warnings.filterwarnings('ignore')
print("Libraries imported successfully.")


Libraries imported successfully.


In [2]:
# 4. CONFIGURATION

PROJECT_ROOT = pathlib.Path.cwd().parent
TFIDF_DIR = PROJECT_ROOT / "data" / "processed" / "tfidf"
MODELS_DIR = PROJECT_ROOT / "models"
BASE_REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR = BASE_REPORTS_DIR / "hyperparameter_tuning"

# Inputs
X_TRAIN_PATH = TFIDF_DIR / "X_train_tfidf.npz"
Y_TRAIN_PATH = TFIDF_DIR / "y_train.csv"

# Cross-Validation Config
CV_SPLITS = 5
SCORING_METRIC = 'f1_macro'
RANDOM_STATE = 42
import os\nMAX_CORES = max(1, int(os.cpu_count() * 0.8))\nprint(f\"Limiting CV tuning to {MAX_CORES} cores to prevent system crash.\")\n
print(f"Artifact Directory: {TFIDF_DIR}")
print(f"Cross Validation: {CV_SPLITS} Folds")
print(f"Scoring Metric: {SCORING_METRIC}")


Artifact Directory: /home/zapp/Kampus/PM-NEW/data/processed/tfidf
Cross Validation: 5 Folds
Scoring Metric: f1_macro


In [3]:
# 5. LOAD TF-IDF ARTIFACTS

if not X_TRAIN_PATH.exists() or not Y_TRAIN_PATH.exists():
    raise FileNotFoundError(f"FAIL: Training artifacts not found. Please run 06_tfidf.ipynb first.")

print("Loading Training Data (Ignoring Val & Test to prevent leakage)...")
X_train = sparse.load_npz(X_TRAIN_PATH)
y_train_df = pd.read_csv(Y_TRAIN_PATH)
y_train = y_train_df.iloc[:, 0]  # Get as Pandas Series

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")


Loading Training Data (Ignoring Val & Test to prevent leakage)...
X_train shape: (30195, 1959527)
y_train shape: (30195,)


In [4]:
# 6. VALIDATE FEATURE AND LABEL ALIGNMENT

if X_train.shape[0] != y_train.shape[0]:
    raise ValueError(f"CRITICAL ERROR: Misalignment! X_train has {X_train.shape[0]} rows, but y_train has {y_train.shape[0]} rows.")

print("Feature and Label alignment validated successfully.")


Feature and Label alignment validated successfully.


In [5]:
# 7. INSPECT TARGET LABELS

class_counts = y_train.value_counts()
class_percentages = y_train.value_counts(normalize=True) * 100

print(f"Number of classes: {len(class_counts)}\n")
print("Target Labels Distribution (Training Data):")
for class_name, count in class_counts.items():
    print(f"- {class_name:<15}: {count} ({class_percentages[class_name]:.2f}%)")
    
# XGBoost mapping logic
label_classes = sorted(y_train.unique())
label_mapping = {label: idx for idx, label in enumerate(label_classes)}
y_train_encoded = y_train.map(label_mapping)


Number of classes: 4

Target Labels Distribution (Training Data):
- normal         : 20192 (66.87%)
- hate_speech    : 5125 (16.97%)
- harassment     : 3659 (12.12%)
- insult         : 1219 (4.04%)


In [6]:
# 8. DEFINE CROSS-VALIDATION STRATEGY

cv_strategy = StratifiedKFold(
    n_splits=CV_SPLITS, 
    shuffle=True, 
    random_state=RANDOM_STATE
)

print(f"Cross-Validation Strategy: StratifiedKFold")
print(f"- Splits: {CV_SPLITS}")
print(f"- Shuffle: True")
print(f"- Random State: {RANDOM_STATE}")
print("\nReasoning: Stratified K-Fold ensures that each fold maintains the same percentage of samples for each cyberbullying class, which is vital for imbalanced datasets.")


Cross-Validation Strategy: StratifiedKFold
- Splits: 5
- Shuffle: True
- Random State: 42

Reasoning: Stratified K-Fold ensures that each fold maintains the same percentage of samples for each cyberbullying class, which is vital for imbalanced datasets.


In [7]:
# 9. DEFINE SCORING STRATEGY

scoring_strategy = SCORING_METRIC

print(f"Primary Scoring Metric: {scoring_strategy}")
print("\nReasoning: Accuracy can be misleading if a majority class dominates. f1_macro calculates the F1-score for each class independently and then takes the unweighted mean. It penalizes models that ignore minority classes.")


Primary Scoring Metric: f1_macro

Reasoning: Accuracy can be misleading if a majority class dominates. f1_macro calculates the F1-score for each class independently and then takes the unweighted mean. It penalizes models that ignore minority classes.


In [8]:
# 10. DEFINE MODEL SEARCH SPACES

# 1. Logistic Regression Grid
param_grid_lr = {
    'C': [0.1, 1.0, 10.0],
    'class_weight': ['balanced', None],
    'solver': ['lbfgs', 'saga'] # 'saga' is good for large sparse datasets
}

# 2. Linear SVM Grid
param_grid_svm = {
    'C': [0.1, 1.0, 10.0],
    'class_weight': ['balanced', None],
    'loss': ['squared_hinge']
}

# 3. XGBoost Randomized Space
# To prevent memory/CPU overload, we use a concise parameter distribution
param_dist_xgb = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 6, 9],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

print("Search spaces defined. Ensuring computationally reasonable grids.")


Search spaces defined. Ensuring computationally reasonable grids.


In [ ]:
# 11. TUNE MODEL 1: LOGISTIC REGRESSION

print("Starting Hyperparameter Tuning for Logistic Regression...")

base_lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)

grid_lr = GridSearchCV(
    estimator=base_lr,
    param_grid=param_grid_lr,
    scoring=scoring_strategy,
    cv=cv_strategy,
    n_jobs=MAX_CORES, pre_dispatch='1.5*n_jobs', # Limit CPU and RAM # Use all CPU cores
    verbose=1
)

start_time = time.time()
grid_lr.fit(X_train, y_train)
lr_duration = time.time() - start_time

best_lr_model = grid_lr.best_estimator_
print(f"\nTuning Completed in {lr_duration:.2f} seconds.")
print(f"Best CV f1_macro score: {grid_lr.best_score_:.4f}")
print(f"Best Parameters: {grid_lr.best_params_}")


Starting Hyperparameter Tuning for Logistic Regression...
Fitting 5 folds for each of 12 candidates, totalling 60 fits


In [ ]:
# 12. TUNE MODEL 2: LINEAR SVM

print("Starting Hyperparameter Tuning for Linear SVM...")

base_svm = LinearSVC(max_iter=2000, random_state=RANDOM_STATE, dual=False)

grid_svm = GridSearchCV(
    estimator=base_svm,
    param_grid=param_grid_svm,
    scoring=scoring_strategy,
    cv=cv_strategy,
    n_jobs=MAX_CORES, pre_dispatch='1.5*n_jobs', # Limit CPU and RAM
    verbose=1
)

start_time = time.time()
grid_svm.fit(X_train, y_train)
svm_duration = time.time() - start_time

best_svm_model = grid_svm.best_estimator_
print(f"\nTuning Completed in {svm_duration:.2f} seconds.")
print(f"Best CV f1_macro score: {grid_svm.best_score_:.4f}")
print(f"Best Parameters: {grid_svm.best_params_}")


In [ ]:
# 13. TUNE REQUIRED BOOSTING MODEL: XGBOOST

print("Starting Hyperparameter Tuning for XGBoost...")
print("Using RandomizedSearchCV to respect computational constraints.")

base_xgb = xgb.XGBClassifier(
    random_state=RANDOM_STATE, 
    use_label_encoder=False, 
    eval_metric='mlogloss',
    tree_method='hist',
    device='cpu' # Reverted to CPU to prevent VRAM OOM # Efficient for sparse data
)

random_xgb = RandomizedSearchCV(
    estimator=base_xgb,
    param_distributions=param_dist_xgb,
    n_iter=30, # Test 10 random combinations
    scoring=scoring_strategy,
    cv=cv_strategy,
    random_state=RANDOM_STATE,
    n_jobs=MAX_CORES, pre_dispatch='1.5*n_jobs', # Limit CPU and RAM\n    verbose=1
)

start_time = time.time()
random_xgb.fit(X_train, y_train_encoded) # Requires integer encoded targets
xgb_duration = time.time() - start_time

best_xgb_model = random_xgb.best_estimator_
print(f"\nTuning Completed in {xgb_duration:.2f} seconds.")
print(f"Best CV f1_macro score: {random_xgb.best_score_:.4f}")
print(f"Best Parameters: {random_xgb.best_params_}")


In [ ]:
# 14. TUNE ADDITIONAL TARGETS

# Target focus is currently `cyberbullying_type`.
print("Additional target models skipped (focusing on primary Cyberbullying Type classification).")


In [ ]:
# 15. COMPARE CROSS-VALIDATION RESULTS

results_summary = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Search Strategy": "GridSearch",
        "Best Parameters": str(grid_lr.best_params_),
        "Best CV F1-Macro": grid_lr.best_score_
    },
    {
        "Model": "Linear SVC",
        "Search Strategy": "GridSearch",
        "Best Parameters": str(grid_svm.best_params_),
        "Best CV F1-Macro": grid_svm.best_score_
    },
    {
        "Model": "XGBoost",
        "Search Strategy": "RandomizedSearch (n_iter=30)",
        "Best Parameters": str(random_xgb.best_params_),
        "Best CV F1-Macro": random_xgb.best_score_
    }
])

results_summary = results_summary.sort_values(by="Best CV F1-Macro", ascending=False).reset_index(drop=True)

print("--- CROSS-VALIDATION RESULTS COMPARISON ---")
display(results_summary)

print("\nNOTE: These are Internal Cross-Validation scores on the Training Set.")
print("Do not declare the final winner until they are tested against the Unseen Test Data in Stage 09.")


In [ ]:
# 16. SAVE BEST TUNED MODELS

# Ensure the best estimators are fully fitted (GridSearch refits automatically on the whole dataset by default)
models_to_save = {
    'logistic_regression_type_tuned': best_lr_model,
    'linear_svm_type_tuned': best_svm_model,
    'xgboost_type_tuned': best_xgb_model
}

for name, model in models_to_save.items():
    model_path = MODELS_DIR / f"{name}.pkl"
    joblib.dump(model, model_path)
    print(f"Saved: {model_path}")


In [ ]:
# 17. SAVE TUNING RESULTS

# Extract raw CV results into DataFrames
lr_cv_results = pd.DataFrame(grid_lr.cv_results_)
svm_cv_results = pd.DataFrame(grid_svm.cv_results_)
xgb_cv_results = pd.DataFrame(random_xgb.cv_results_)

# Save to CSV
lr_cv_results.to_csv(REPORTS_DIR / "cv_results_logistic_regression.csv", index=False)
svm_cv_results.to_csv(REPORTS_DIR / "cv_results_linear_svm.csv", index=False)
xgb_cv_results.to_csv(REPORTS_DIR / "cv_results_xgboost.csv", index=False)

print("Detailed Cross-Validation results saved to reports/.")


In [ ]:
# 18. SAVE TUNING METADATA

tuning_metadata = {
    "random_state": RANDOM_STATE,
    "cv_strategy": "StratifiedKFold",
    "cv_splits": CV_SPLITS,
    "scoring_metric": SCORING_METRIC,
    "logistic_regression": {
        "best_params": grid_lr.best_params_,
        "best_cv_score": float(grid_lr.best_score_)
    },
    "linear_svm": {
        "best_params": grid_svm.best_params_,
        "best_cv_score": float(grid_svm.best_score_)
    },
    "xgboost": {
        "best_params": random_xgb.best_params_,
        "best_cv_score": float(random_xgb.best_score_)
    }
}

meta_path = REPORTS_DIR / "hyperparameter_tuning_metadata.json"
with open(meta_path, 'w') as f:
    json.dump(tuning_metadata, f, indent=4)

print(f"Tuning metadata saved to {meta_path}")


# 19. TUNING SUMMARY

### Models Tuned
1. **Logistic Regression** (Target: `cyberbullying_type`)
2. **Linear SVM** (Target: `cyberbullying_type`)
3. **XGBoost** (Target: `cyberbullying_type`)

### Search Strategy
- **Metric**: `f1_macro` (chosen to penalize poor performance on minority classes).
- **Validation**: 5-Fold Stratified Cross-Validation on `X_train_tfidf`.
- **Method**: Exhaustive `GridSearchCV` for LR and SVM; efficient `RandomizedSearchCV` (10 iters) for XGBoost to respect computing bounds.

### Results
The best estimators and their respective scores have been successfully logged. The `GridSearchCV` object automatically refits the best found parameters on the entire `X_train` dataset before yielding `best_estimator_`.

### Saved Artifacts
- **Models**: `*_type_tuned.pkl` saved in `models/`
- **Logs**: `cv_results_*.csv` and `hyperparameter_tuning_metadata.json` saved in `reports/`

### Next Step
We now possess both Baseline models and Tuned models. The final and most crucial step is to test all of them against the strictly isolated Validation and Test datasets in `notebooks/09_model_evaluation.ipynb`.


# 20. HOW TO RUN THIS NOTEBOOK

1. Activate the project's virtual environment.
2. Ensure the TF-IDF artifacts exist in: `data/processed/tfidf/` (specifically `X_train_tfidf.npz`).
3. Ensure the baseline model training stage has been completed: `notebooks/07_model_training.ipynb`.
4. Open: `notebooks/08_hyperparameter_tuning.ipynb`
5. Verify the selected search spaces in **Step 10**. You may expand or reduce them based on your machine's CPU capacity.
6. Run the notebook from the first cell to the last cell.
7. **WARNING:** Hyperparameter tuning is computationally heavy. **Step 11, 12, and especially 13** may take anywhere from a few minutes to an hour depending on your hardware. Let it run without interrupting the kernel.
8. Review the final comparison table in **Step 15**.
9. Confirm the generation of the `_tuned.pkl` models in the `models/` directory.
10. Proceed to: `notebooks/09_model_evaluation.ipynb` to evaluate and visualize model performance.
